# Demo 05. Least Squares and Orthogonal Polynomials

**Standard 4** (least squares), Weeks 3-4.

Interpolation forces the curve through every point; least squares finds the closest fit within a lower-dimensional family, which is appropriate for noisy data. This demo treats the discrete problem, fitting noisy samples with two solvers, and the continuous problem, best $L^2$ approximation of a function, where orthogonal polynomials reduce a coupled system to independent projections.

In [ ]:
# --- Colab / Jupyter setup ------------------------------------------------
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from ipywidgets import interact, IntSlider, FloatSlider

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 5)

## Orthogonality

The least-squares coefficients solve the normal equations $Gc = b$ with Gram matrix $G_{jk} = \langle \phi_j, \phi_k\rangle$. In the monomial basis the Gram matrix is dense and severely ill-conditioned; on $[0,1]$ it is exactly the Hilbert matrix. In an orthogonal basis, $\langle P_j, P_k\rangle = 0$ for $j\neq k$, so $G$ is diagonal and each coefficient is an independent projection

$$ c_k = \frac{\langle f, P_k\rangle}{\langle P_k, P_k\rangle}. $$

The discrete analogue is to solve via QR rather than the normal equations.

In [ ]:
# ---------------------------------------------------------------------------
# The design matrix, and the data the puzzles run on
# ---------------------------------------------------------------------------
# Fitting a degree-m polynomial to (x_i, y_i) means choosing coefficients c so
# that V c is as close to y as possible, where V is the design matrix holding
# each basis function evaluated at each data point. For the monomial basis
# 1, x, x^2, ... that is the Vandermonde matrix.

def vandermonde(x, m):
    """Design matrix for the monomials, V[i][j] = x_i^j."""
    x = np.asarray(x, float)
    return np.vander(x, m + 1, increasing=True)

def poly_eval(coeffs, x):
    """Evaluate a polynomial given coefficients [c0, c1, ...] (low -> high)."""
    x = np.asarray(x, float)
    return sum(c * x**k for k, c in enumerate(coeffs))

X_DEMO = np.array([-1.0, -0.5, 0.0, 0.5, 1.0])
Y_DEMO = np.array([2.0, 1.0, 3.0, 5.0, 9.0])

print("x =", X_DEMO)
print("y =", Y_DEMO)
print("V =")
print(vandermonde(X_DEMO, 2))

In [ ]:
# ---------------------------------------------------------------------------
# Route one: form the normal equations and solve them
# ---------------------------------------------------------------------------
# Setting the derivative of ||Vc - y||^2 to zero gives (V^T V) c = V^T y. The
# matrix G = V^T V is the Gram matrix of the basis: G[j][k] is the inner
# product of basis column j with basis column k. Written as loops rather than
# as matrix products, so every entry says where it came from.

def fit_normal(V, y, n, m):
    """Least-squares coefficients from the normal equations G c = b."""
    G = np.zeros((m + 1, m + 1))
    b = np.zeros(m + 1)
    for j in range(m + 1):
        for i in range(n):
            b[j] = b[j] + V[i][j] * y[i]
        for k in range(m + 1):
            for i in range(n):
                G[j][k] = G[j][k] + V[i][j] * V[i][k]
    return np.linalg.solve(G, b)

V_DEMO = vandermonde(X_DEMO, 2)
print("G = V^T V is dense: every basis function overlaps every other")
print(V_DEMO.T @ V_DEMO)
print("\ncoefficients in the monomial basis:", fit_normal(V_DEMO, Y_DEMO, 5, 2))

In [ ]:
# ---------------------------------------------------------------------------
# Making the basis orthogonal
# ---------------------------------------------------------------------------
# Gram-Schmidt, applied to the columns of V under the discrete inner product
# <u, v> = sum_i u_i v_i. Each new column has its projection onto the earlier
# ones removed, which leaves a basis spanning the same polynomials with every
# pair of columns orthogonal.

def orthogonal_columns(V):
    """Gram-Schmidt on the columns of V. Returns a matrix with the same span."""
    Q = np.zeros_like(V, dtype=float)
    for j in range(V.shape[1]):
        v = V[:, j].astype(float).copy()
        for k in range(j):
            v = v - (V[:, j] @ Q[:, k]) / (Q[:, k] @ Q[:, k]) * Q[:, k]
        Q[:, j] = v
    return Q

P_DEMO = orthogonal_columns(V_DEMO)
print("the orthogonal basis at these five nodes (columns P0, P1, P2):")
print(P_DEMO)
print("\nas polynomials: P0 = 1,  P1 = x,  P2 = x^2 - 1/2")
print("\nits Gram matrix, which is now diagonal:")
print(np.round(P_DEMO.T @ P_DEMO, 12))

In [ ]:
# ---------------------------------------------------------------------------
# Route two: project onto the orthogonal basis
# ---------------------------------------------------------------------------
# With an orthogonal basis the Gram matrix is diagonal, so the normal equations
# fall apart into m+1 independent one-line divisions. There is no system left
# to solve.

def fit_orthogonal(P, y, n, m):
    """Least-squares coefficients by independent projection onto each column."""
    c = np.zeros(m + 1)
    for k in range(m + 1):
        num = 0.0
        den = 0.0
        for i in range(n):
            num = num + P[i][k] * y[i]
            den = den + P[i][k] * P[i][k]
        c[k] = num / den
    return c

c_orth = fit_orthogonal(P_DEMO, Y_DEMO, 5, 2)
c_mono = fit_normal(V_DEMO, Y_DEMO, 5, 2)
print("coefficients in the orthogonal basis:", c_orth)
print("coefficients in the monomial basis  :", c_mono)
print("\nDifferent numbers, because they are coefficients in different bases.")
print("Same fitted polynomial:", np.allclose(P_DEMO @ c_orth, V_DEMO @ c_mono))

In [ ]:
# ---------------------------------------------------------------------------
# What the monomial basis costs
# ---------------------------------------------------------------------------
# Squaring a matrix squares its condition number, so forming V^T V throws away
# half the available digits before the solve begins. QR factorises V itself and
# never forms the square. Watch the two answers separate as the degree rises.

rng = np.random.default_rng(0)
xs = np.linspace(-1.0, 1.0, 25)
ys = np.sin(2 * xs) + 0.15 * rng.standard_normal(25)

def fit_qr(V, y):
    """Least squares via QR, which never forms V^T V."""
    Q, R = np.linalg.qr(V)
    return np.linalg.solve(R, Q.T @ y)

print(f"{'degree':>7} {'cond(V)':>11} {'cond(V^T V)':>13} {'|c_normal - c_QR|':>19}")
for deg in (3, 6, 9, 12, 15):
    V = vandermonde(xs, deg)
    c1 = fit_normal(V, ys, len(xs), deg)
    c2 = fit_qr(V, ys)
    print(f"{deg:7d} {np.linalg.cond(V):11.3e} {np.linalg.cond(V.T @ V):13.3e}"
          f" {np.linalg.norm(c1 - c2):19.3e}")

V9 = vandermonde(xs, 9)
print(f"\nat degree 9:  cond(V)^2 = {np.linalg.cond(V9)**2:.3e}"
      f"   cond(V^T V) = {np.linalg.cond(V9.T @ V9):.3e}")
Q9 = orthogonal_columns(V9)
G9, GQ = V9.T @ V9, Q9.T @ Q9
print(f"largest off-diagonal entry, monomial Gram matrix   : "
      f"{np.abs(G9 - np.diag(np.diag(G9))).max():.2e}")
print(f"largest off-diagonal entry, orthogonal Gram matrix : "
      f"{np.abs(GQ - np.diag(np.diag(GQ))).max():.2e}")

In [ ]:
# ---------------------------------------------------------------------------
# The fit itself, on noisy data
# ---------------------------------------------------------------------------
def show_fit(degree=3):
    """Fit the noisy samples and draw the result against the true signal."""
    V = vandermonde(xs, degree)
    c = fit_qr(V, ys)
    xx = np.linspace(-1, 1, 300)
    plt.figure()
    plt.plot(xs, ys, "b.", ms=8, label="noisy data")
    plt.plot(xx, np.sin(2 * xx), "k-", lw=1, alpha=0.5, label="true signal")
    plt.plot(xx, poly_eval(c, xx), "r-", lw=2, label=f"least-squares fit (deg {degree})")
    plt.legend(); plt.xlabel("x"); plt.title("Discrete least-squares fit")
    plt.show()

interact(show_fit, degree=IntSlider(min=1, max=12, step=1, value=3,
                                    description="degree"));

## Continuous least squares (symbolic)

In [ ]:
# ---------------------------------------------------------------------------
# Continuous least squares & orthogonal polynomials  (symbolic, via SymPy)
# ---------------------------------------------------------------------------
# Best L2 approximation of a function on [-1, 1] means minimising the integral
#   integral_{-1}^{1} (f(x) - p(x))^2 dx.
# In the monomial basis this couples all coefficients through a dense, ill-conditioned Gram
# matrix (the Hilbert matrix on [0,1]). In an orthogonal basis it decouples: each coefficient is an independent
# projection. We build the Legendre polynomials by Gram-Schmidt on {1, x, x^2,...}
# under the inner product <p, q> = integral_{-1}^1 p q dx.

x = sp.symbols("x")

def legendre_basis(n):
    """Return orthogonal polynomials P_0..P_n on [-1,1] via Gram-Schmidt.

    Orthogonalises the monomials 1, x, x^2, ... under <p,q> = int_{-1}^1 p q dx.
    (These are the Legendre polynomials up to normalisation.)
    """
    def inner(p, q):
        return sp.integrate(p * q, (x, -1, 1))
    basis = []
    for k in range(n + 1):
        p = x**k
        for b in basis:                     # subtract projections onto previous
            p = p - inner(x**k, b) / inner(b, b) * b
        basis.append(sp.expand(p))
    return basis

def best_l2_approx(f_expr, n):
    """Best degree-n L2 approximation of f on [-1,1] using the orthogonal basis.

    Because the basis is orthogonal, coefficient k is just <f, P_k>/<P_k, P_k> --
    computed independently, no linear system to solve.
    """
    basis = legendre_basis(n)
    approx = 0
    for b in basis:
        coeff = sp.integrate(f_expr * b, (x, -1, 1)) / sp.integrate(b * b, (x, -1, 1))
        approx += coeff * b
    return sp.expand(approx), basis

# Example: approximate exp(x) by a quadratic and show the orthogonal basis.
approx, basis = best_l2_approx(sp.exp(x), 2)
print("Orthogonal (Legendre) basis P0..P2:")
for k, b in enumerate(basis):
    print(f"  P{k} =", b)
print("\nBest quadratic L2 approximation of exp(x) on [-1,1]:")
sp.pprint(approx)

## Summary

- Least squares minimizes the residual in the 2-norm, and the minimiser satisfies the normal equations $Gc = b$ with $G_{jk} = \langle \phi_j, \phi_k\rangle$.
- In the monomial basis $G$ is dense, because no two monomials are orthogonal, and it is badly conditioned. Forming $G = V^{\top}V$ squares the condition number of $V$, which is why the normal equations and QR give visibly different coefficients by degree 12.
- In an orthogonal basis $G$ is diagonal and the system falls apart into independent projections $c_k = \langle f, P_k\rangle / \langle P_k, P_k\rangle$. No linear system is solved at all.
- Gram-Schmidt on the columns of $V$ produces such a basis for the discrete problem; the same construction under $\langle p,q\rangle = \int_{-1}^{1} pq\,dx$ produces the Legendre polynomials.
- These orthogonal polynomials reappear in demo 07 for Gaussian quadrature.